# Training TinyViT with Sparsity

In [1]:
import torch
import torch.nn as nn
import torch.optim
import torch.utils.data

import numpy as np
import os
import time

from import_shelf import shelf
from shelf.models.transformer import VisionTransformer
from shelf.dataloaders.cifar import get_CIFAR10_dataset
from shelf.trainers import train, validate, gradient_fo

from tqdm import tqdm


### HYPERPARAMS ###

EPOCHS = 500
BATCH_SIZE = 512
LEARNING_RATE = 1e-4

IMAGE_SIZE = 32
PATCH_SIZE = 4
DIM_HIDDEN = 256
DEPTH = 4
NUM_HEADS = 6
DIM_MLP = 256
DROPOUT = 0.1
EMB_DROPOUT = 0.1

NUM_CLASSES = 10

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PATH_MODEL = './saves/train_tinyvit/model.pth'


### DATA LOADING ###

train_loader, val_loader = get_CIFAR10_dataset(batch_size=BATCH_SIZE)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')


### MODEL ###

model = VisionTransformer(
    image_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    dim=DIM_HIDDEN,
    depth=DEPTH,
    heads=NUM_HEADS,
    mlp_dim=DIM_MLP,
    dropout=0.1,
    emb_dropout=0.1,
    num_classes=NUM_CLASSES,
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of parameters: {num_params}")


### OTHERS ###

criterion = nn.CrossEntropyLoss()
sparsifier = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS, eta_min=5e-6)


Files already downloaded and verified
Number of parameters: 2136842


In [2]:

### TRAINING ###

train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

for epoch in range(EPOCHS):
    start_time = time.time()
    
    train_acc, train_loss = 0, 0

    # train
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss_sparsify = 0
        for p in model.parameters():
            loss_sparsify += sparsifier(p, torch.zeros_like(p))

        sparsifying_rate = 0
        loss += sparsifying_rate * loss_sparsify

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_acc += (outputs.argmax(1) == labels).float().mean().item()

    val_acc, val_loss = validate(val_loader, model, criterion, epoch, verbose=False)

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)


    # measure decay rate of parameters

    images, lables = next(iter(train_loader))
    real_gradient = {name: param.grad for name, param in model.named_parameters()}
    real_gradient_flatten = torch.cat([g.flatten() for g in real_gradient.values()]).abs()
    real_gradient_flatten = torch.sort(real_gradient_flatten, descending=True).values

    real_gradient_flatten_log = torch.log(real_gradient_flatten + 1e-99).cpu()
    # linear fit
    x = torch.arange(len(real_gradient_flatten_log)).float()
    A = torch.vstack([x, torch.ones_like(x)]).T
    m, c = torch.linalg.lstsq(A, real_gradient_flatten_log).solution

    print(
        f"Epoch {epoch+1:3d}/{EPOCHS}, "
        f"LR: {scheduler.get_last_lr()[0]:.4e} | "
        f"Train Loss: {train_loss:.4f}, "
        f"Train Acc: {train_acc:.2f}%, "
        f"Val Loss: {val_loss:.4f}, "
        f"Val Acc: {val_acc*100:.2f}%, "
        f"Decay rate: {m.item():.4e} | "
        f"Time: {time.time() - start_time:.3f}s"
    )
    
    scheduler.step()


torch.save(model.state_dict(), PATH_MODEL)

print(f"Model saved to {PATH_MODEL}")

Epoch   1/500, LR: 1.0000e-04 | Train Loss: 215.3213, Train Acc: 17.31%, Val Loss: 1.9149, Val Acc: 29.44%, Decay rate: -7.1547e-06 | Time: 17.711s
Epoch   2/500, LR: 9.9999e-05 | Train Loss: 196.9629, Train Acc: 26.42%, Val Loss: 1.7005, Val Acc: 39.31%, Decay rate: -6.7838e-06 | Time: 17.868s
Epoch   3/500, LR: 9.9996e-05 | Train Loss: 185.3402, Train Acc: 31.17%, Val Loss: 1.5773, Val Acc: 43.84%, Decay rate: -6.4152e-06 | Time: 17.843s
Epoch   4/500, LR: 9.9992e-05 | Train Loss: 178.5291, Train Acc: 34.05%, Val Loss: 1.5136, Val Acc: 46.16%, Decay rate: -6.2581e-06 | Time: 17.775s
Epoch   5/500, LR: 9.9985e-05 | Train Loss: 172.6294, Train Acc: 36.19%, Val Loss: 1.4380, Val Acc: 49.57%, Decay rate: -6.2270e-06 | Time: 17.900s
Epoch   6/500, LR: 9.9977e-05 | Train Loss: 169.3173, Train Acc: 37.47%, Val Loss: 1.3809, Val Acc: 51.21%, Decay rate: -6.0265e-06 | Time: 18.058s
Epoch   7/500, LR: 9.9966e-05 | Train Loss: 165.1450, Train Acc: 39.07%, Val Loss: 1.3531, Val Acc: 51.97%, Deca

KeyboardInterrupt: 